# **Вмешательство: ablation, occlusion и RISE на одной картинке**

Практика к модулю [«Объяснение DL моделей через вмешательство»](https://ai-interpretability.school).

Все три метода отвечают на один вопрос — «что будет с прогнозом, если часть входа убрать» —
и отличаются только тем, **что именно** они убирают и **как** собирают ответы в карту.
Здесь мы строим их сами, на одной фотографии и одной модели.

А потом делаем то, ради чего практика и написана: **сравниваем их числом, а не глазами**.
И обнаруживаем, что верность карты зависит не только от карты: поменяв в метрике одно слово,
можно вывести в лидеры случайный шум.

In [ ]:
import torch
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image, ImageFilter
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0)
np.random.seed(0)

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

RAW = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/'

def load(name):
    return Image.open(BytesIO(requests.get(RAW + name).content)).convert('RGB')

image = load('pig.png').resize((224, 224))
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get(RAW + 'imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    probs = model(x).softmax(1)
target = probs.argmax().item()
print(f'Класс: {target} — {categories[target]}, уверенность {probs[0, target]:.3f}')

## 1. Baseline: то, чем мы заменяем убранное

У всех трёх методов есть общая деталь, о которой урок говорит одной строкой, а на практике
она решает всё: **чем заменяется убранная часть входа**. Нулём? Каким нулём — чёрным пикселем
или нулём в нормализованных координатах?

Это тот же вопрос, что и baseline у Integrated Gradients, и та же ловушка: `torch.zeros_like(x)`
после нормализации — **не чёрный**, а серый. Проверим.

In [ ]:
black = transform(Image.new('RGB', (224, 224), (0, 0, 0)))
blur = transform(image.filter(ImageFilter.GaussianBlur(12)))

print(f'нулевой тензор   → среднее по каналам: {(x * 0).mean():.4f}')
print(f'настоящий чёрный → те же координаты:   {black.mean():.4f}')

def predict(batch):
    with torch.no_grad():
        return model(batch).softmax(1)[:, target]

print(f'\nуверенность на нулевом тензоре: {predict(torch.zeros_like(x)).item():.4f}')
print(f'уверенность на чёрном кадре:    {predict(black.unsqueeze(0)).item():.4f}')
print(f'уверенность на размытии:        {predict(blur.unsqueeze(0)).item():.4f}')

# Размытие как baseline: оно убирает детали, но не создаёт на их месте объект,
# которого в природе не бывает. Чёрный прямоугольник посреди фотографии — создаёт.
BASELINE = blur
base_score = predict(x).item()

## 2. Feature Ablation

Формула из урока: $AblImp_S(x) = f(x) - f(x^{(S\to b)})$. Для изображения «признак» — это
пиксель, и первый соблазн посчитать так по каждому. Посчитаем.

In [ ]:
# Абляция по одиночным пикселям: считаем на грубой сетке, иначе это 50 176 прогонов.
step = 16
coords = [(i, j) for i in range(0, 224, step) for j in range(0, 224, step)]
batch = x.repeat(len(coords), 1, 1, 1)
for k, (i, j) in enumerate(coords):
    batch[k, :, i, j] = BASELINE[:, i, j]

scores = torch.cat([predict(batch[s:s + 64]) for s in range(0, len(batch), 64)])
single = base_score - scores
print(f'уверенность модели:      {base_score:.4f}')
print(f'максимум по одному пикселю: {single.max():.6f} — {single.max() / base_score:.2%} уверенности')
print(f'среднее по одному пикселю:  {single.mean():.6f} — {single.mean() / base_score:.3%}')

Средний вклад одного пикселя — тысячные доли процента, причём со знаком минус: убрать
случайный пиксель в среднем скорее слегка повышает уверенность. Максимум — пятая часть
процента. Это не «пиксели не важны», это **насыщение**: сеть без труда
восстанавливает один потерянный пиксель по соседям, и разность двух почти одинаковых чисел
тонет в шуме.

Отсюда практическое правило урока: для изображений вмешательство всегда **групповое**.
Повторим блоками $16\times16$.

In [ ]:
def ablate_regions(size):
    """Абляция непересекающимися блоками size x size — по одному прогону на блок."""
    cells = [(i, j) for i in range(0, 224, size) for j in range(0, 224, size)]
    batch = x.repeat(len(cells), 1, 1, 1)
    for k, (i, j) in enumerate(cells):
        batch[k, :, i:i + size, j:j + size] = BASELINE[:, i:i + size, j:j + size]
    scores = torch.cat([predict(batch[s:s + 32]) for s in range(0, len(batch), 32)])
    heat = torch.zeros(224, 224)
    for (i, j), s in zip(cells, scores):
        heat[i:i + size, j:j + size] = base_score - s
    return heat

abl = ablate_regions(16)
print(f'абляция блока 16x16: максимум {abl.max():.4f} — {abl.max() / base_score:.1%} уверенности')

## 3. Occlusion

Отличие от абляции ровно одно: окно **скользит с перекрытием**, и вклад пикселя усредняется
по всем окнам, которые его накрыли — формула урока
$Occ_i(x)=\frac1k\sum_j\big(f(x)-f(x^{(R_j\to b)})\big)$.

In [ ]:
def occlusion(size, stride):
    """Скользящее окно с перекрытием; вклад пикселя — среднее по накрывшим его окнам."""
    cells = [(i, j) for i in range(0, 224 - size + 1, stride)
                    for j in range(0, 224 - size + 1, stride)]
    batch = x.repeat(len(cells), 1, 1, 1)
    for k, (i, j) in enumerate(cells):
        batch[k, :, i:i + size, j:j + size] = BASELINE[:, i:i + size, j:j + size]
    scores = torch.cat([predict(batch[s:s + 32]) for s in range(0, len(batch), 32)])

    total, count = torch.zeros(224, 224), torch.zeros(224, 224)
    for (i, j), s in zip(cells, scores):
        total[i:i + size, j:j + size] += base_score - s
        count[i:i + size, j:j + size] += 1
    return total / count, len(cells)

occ_64, n64 = occlusion(64, 16)
occ_32, n32 = occlusion(32, 16)
print(f'окно 64, шаг 16: {n64} прогонов сети')
print(f'окно 32, шаг 16: {n32} прогонов сети')

## 4. RISE

Три шага из урока: маленькая бернуллиевская сетка $h\times w$, апсемплинг билинейной
интерполяцией со случайным сдвигом, взвешенное усреднение масок. Нормировка — та самая
$\frac{1}{N\cdot\mathbb{E}[M]}$, где $\mathbb{E}[M]=p$.

In [ ]:
def rise(n_masks, p=0.5, s=7, batch_size=50):
    """RISE: сетка s x s из Бернулли → апсемплинг → случайный сдвиг → кроп до 224."""
    cell = int(np.ceil(224 / s))
    up = (s + 1) * cell
    heat = torch.zeros(224, 224)
    done = 0
    while done < n_masks:
        k = min(batch_size, n_masks - done)
        grid = (torch.rand(k, 1, s, s) < p).float()
        big = torch.nn.functional.interpolate(grid, size=(up, up),
                                              mode='bilinear', align_corners=False)
        masks = torch.empty(k, 1, 224, 224)
        for m in range(k):
            di, dj = np.random.randint(0, cell, 2)
            masks[m] = big[m, :, di:di + 224, dj:dj + 224]
        scores = predict(x * masks)
        heat += (masks[:, 0] * scores[:, None, None]).sum(0)
        done += k
    return heat / (n_masks * p)

rise_50 = rise(50)
rise_200 = rise(200)
rise_500 = rise(500)
print('RISE посчитан для 50, 200 и 500 масок')

In [ ]:
def show(ax, heat, title):
    ax.imshow(image)
    ax.imshow(heat.numpy(), cmap='jet', alpha=0.5)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

MAPS = {'Ablation, блоки 16': abl, 'Occlusion, окно 64': occ_64, 'Occlusion, окно 32': occ_32,
        'RISE, 50 масок': rise_50, 'RISE, 200 масок': rise_200, 'RISE, 500 масок': rise_500}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, heat) in zip(axes.flat, MAPS.items()):
    show(ax, heat, name)
plt.tight_layout()
plt.show()

## 5. Кто из них прав

Глазами карты разные, и какая-то наверняка кажется убедительнее. Но «убедительнее» — не
«вернее»: об этом весь блок про оценку объяснений. Померим двумя стандартными метриками.

**Deletion.** Убираем пиксели по убыванию важности и смотрим, как падает уверенность.
Чем **ниже** площадь под кривой, тем лучше.

**Insertion.** Наоборот: стартуем с размытой картинки и возвращаем пиксели по убыванию
важности. Чем **выше** площадь, тем лучше.

Добавим в сравнение случайную карту — она должна оказаться худшей по обеим метрикам.

In [ ]:
def curve_auc(heat, mode, steps=100):
    """deletion: из картинки убираем; insertion: в размытие возвращаем."""
    order = heat.flatten().argsort(descending=True)
    per = len(order) // steps
    cur = (x if mode == 'deletion' else BASELINE.unsqueeze(0)).clone()
    src = BASELINE if mode == 'deletion' else x[0]
    curve = [predict(cur).item() / base_score]
    for t in range(steps):
        idx = order[t * per:(t + 1) * per]
        rows, cols = idx // 224, idx % 224
        cur[0, :, rows, cols] = src[:, rows, cols]
        curve.append(predict(cur).item() / base_score)
    return float(np.trapezoid(curve, dx=1 / steps)), curve

everything = dict(MAPS, **{'случайная карта': torch.rand(224, 224)})
report = {}
for name, heat in everything.items():
    d, dc = curve_auc(heat, 'deletion')
    i, ic = curve_auc(heat, 'insertion')
    report[name] = (d, i, dc, ic)
    print(f'{name:20} deletion {d:.4f} (ниже — лучше)   insertion {i:.4f} (выше — лучше)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
grid = np.linspace(0, 1, 101)
for name, (d, i, dc, ic) in report.items():
    style = dict(lw=2.5, color='crimson') if name == 'случайная карта' else dict(lw=1.2)
    ax1.plot(grid, dc, label=f'{name} ({d:.3f})', **style)
    ax2.plot(grid, ic, label=f'{name} ({i:.3f})', **style)
ax1.set_title('Deletion: ниже кривая — лучше карта')
ax2.set_title('Insertion: выше кривая — лучше карта')
for ax in (ax1, ax2):
    ax.set_xlabel('доля изменённых пикселей')
    ax.set_ylabel('уверенность, доля от исходной')
    ax.grid(alpha=.3)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

best_del = min(report, key=lambda k: report[k][0])
best_ins = max(report, key=lambda k: report[k][1])
print(f'лучшая по deletion:  {best_del}')
print(f'лучшая по insertion: {best_ins}')

## 6. Метрика зависит от того, чем «удалять»

Обе метрики согласились. Но в них спрятан тот же параметр, что и в самих методах, — **чем
заменяется убранное**. Мы взяли размытие. Возьмём чёрный и пересчитаем deletion, ничего
больше не меняя.

In [ ]:
def deletion_with(heat, filler, steps=100):
    order = heat.flatten().argsort(descending=True)
    per = len(order) // steps
    cur = x.clone()
    curve = [1.0]
    for t in range(steps):
        idx = order[t * per:(t + 1) * per]
        rows, cols = idx // 224, idx % 224
        cur[0, :, rows, cols] = filler[:, rows, cols]
        curve.append(predict(cur).item() / base_score)
    return float(np.trapezoid(curve, dx=1 / steps))

print(f'{"":22}{"размытие":>12}{"чёрный":>12}')
for name in ('Occlusion, окно 64', 'RISE, 200 масок', 'случайная карта'):
    h = everything[name]
    print(f'{name:22}{deletion_with(h, blur):>12.4f}{deletion_with(h, black):>12.4f}')

Со сменой одного слова в определении метрики случайная карта из явного аутсайдера
превращается в победительницу: она обгоняет и RISE, и occlusion — последний почти на порядок.

Причина не в картах, а в самом вмешательстве. Чёрный прямоугольник посреди фотографии — это
объект, которого в обучающей выборке не было; случайная карта разбрасывает такие пятна
равномерно и превращает вход в соль-перец шум, ломающий сразу все текстуры. Уверенность падает
стремительно, и метрика честно записывает это как «карта хорошая».

То есть deletion меряет не только верность карты, но и **сдвиг распределения**, который вносит
само удаление. Это ровно та критика, из-за которой придумали ROAR и её родню, и ровно та
причина, по которой в блоке про оценку объяснений мы не пользуемся одной метрикой.

**Задание 1.** Отсортируйте методы по insertion. Совпал ли порядок с вашим впечатлением от
картинок в разделе 4?

**Задание 2.** Пересоберите карты, взяв `BASELINE = black` в самих методах (а не только
в метрике). Насколько изменится occlusion? А RISE, который чёрный baseline не использует вовсе?

**Задание 3.** Проверьте, как insertion меняется, если удалять не по одному пикселю, а блоками
$8\times8$ (подсказка: усредните карту через `avg_pool2d` перед сортировкой). Случайная карта
остаётся худшей?

## 7. Что делает параметр $p$

В уроке сказано: «низкое $p$ обнуляет слишком много пикселей, высокое $p$ почти не меняет
картинку». Посмотрим на размах карты — если он схлопывается, метод перестал различать
области, даже если формально отработал.

In [ ]:
print(f'{"p":>5}{"insertion":>12}{"размах карты":>16}')
for p in (0.1, 0.3, 0.5, 0.7, 0.9):
    heat = rise(200, p=p)
    ins, _ = curve_auc(heat, 'insertion')
    print(f'{p:>5}{ins:>12.4f}{(heat.max() - heat.min()).item():>16.4f}')

**Задание 4.** При каком $p$ размах максимален? Совпадает ли он с $p$, при котором
максимален insertion? Если нет — какая из двух величин важнее для читательницы отчёта?

**Задание 5.** Occlusion с окном 32 требует больше прогонов, чем с окном 64. Посчитайте,
во сколько раз, и сравните с выигрышем в insertion. Стоило ли оно того?

## Что стоит унести

- **Все три метода — одна формула** $f(x) - f(x^{(S\to b)})$ с разным $S$: одиночный пиксель
  у наивной абляции, скользящее окно у occlusion, случайная маска у RISE.
- **По одному пикселю не работает.** Вклад одиночного пикселя — сотые доли процента: сеть
  восстанавливает его по соседям. Вмешательство для изображений всегда групповое.
- **Baseline — часть метода, а не деталь реализации.** Та же ловушка с серым `zeros_like`,
  что у Integrated Gradients, и тот же выбор между чёрным и размытием.
- **Метрика верности сама может быть неверной.** Deletion с агрессивным baseline присуждает
  первое место случайной карте, потому что меряет реакцию сети на артефакт вмешательства,
  а не важность пикселей. Одной метрики не хватает — нужны две, и они должны согласоваться.
- **Цена.** Occlusion 32/16 — это сотни прогонов сети, RISE 500 — пятьсот. Градиентные методы
  обходятся одним обратным проходом. За независимость от внутренностей модели платят временем.

**Метки:** локальные, post-hoc, model-agnostic — нужны только вход и выход. Вход: модель как
чёрный ящик и одно изображение. Выход: карта важности размера входа.